# C7-cnn-transfer — Practice p24 — Solution

**Type:** constrained coding · **Difficulty:** core · **Concepts:** tensor-shape-tracing, cnn-training

The floor formula is applied independently to height and width. The committed large-input trace is `(2,13,106,115) → (2,13,53,58) → (2,23,53,27)`. On the training crop, the same layers give `(12,13,16,18) → (12,13,8,9) → (12,23,8,3)` before adaptive pooling and the classifier.

## Part I — committed hand trace

In [ ]:
shape_after_a = (2, 13, 106, 115)
shape_after_pool = (2, 13, 53, 58)
shape_after_b = (2, 23, 53, 27)
shape_trace = [shape_after_a, shape_after_pool, shape_after_b]

### MARKED VERIFICATION CELL

Run only after committing Part I. It reports one agreement bit and never observed shapes.

In [ ]:
import torch
from torch import nn

committed_named_shapes = [shape_after_a, shape_after_pool, shape_after_b]
if any(type(shape) is not tuple or len(shape) != 4 for shape in committed_named_shapes):
    raise RuntimeError("commit each named shape as an exact four-entry tuple")
if type(shape_trace) is not list or shape_trace != committed_named_shapes:
    raise RuntimeError("shape_trace must equal the three named shape tuples in order")

verification_stack = nn.Sequential(
    nn.Conv2d(7, 13, kernel_size=5, stride=2, padding=2),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
    nn.Conv2d(13, 23, kernel_size=(3, 5), stride=(1, 2), padding=(1, 0)),
)
verify_x = torch.zeros(2, 7, 211, 229)
observed = []
for layer in verification_stack:
    verify_x = layer(verify_x)
    observed.append(tuple(verify_x.shape))
trace_ok = committed_named_shapes == observed and shape_trace == observed
print("trace_ok:", trace_ok)
del observed, verify_x, verification_stack, committed_named_shapes

## Part II — construct and train the same stack

In [ ]:
if trace_ok is not True:
    raise RuntimeError("Part I must be committed and correct before Part II")

torch.set_default_dtype(torch.float64)
SEED = 20260804
torch.manual_seed(SEED)
generator = torch.Generator(device="cpu").manual_seed(SEED)
train_X = 0.04 * torch.randn(12, 7, 31, 35, generator=generator)
train_y = torch.arange(12, dtype=torch.long) % 4
train_X[train_y == 0, :, :, 3:8] += 0.8
train_X[train_y == 1, :, 10:15, :] += 0.8
train_X[train_y == 2, :, :, 20:25] -= 0.8
train_X[train_y == 3, :, 20:25, :] -= 0.8

In [ ]:
crop_trace = [
    (12, 13, 16, 18),
    (12, 13, 8, 9),
    (12, 23, 8, 3),
]

features = nn.Sequential(
    nn.Conv2d(7, 13, kernel_size=5, stride=2, padding=2),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
    nn.Conv2d(13, 23, kernel_size=(3, 5), stride=(1, 2), padding=(1, 0)),
)
model = nn.Sequential(
    features,
    nn.AdaptiveAvgPool2d((2, 2)),
    nn.Flatten(1),
    nn.Linear(23 * 2 * 2, 4),
)

constructed_shapes = []
with torch.no_grad():
    traced_value = train_X
    for layer in features:
        traced_value = layer(traced_value)
        constructed_shapes.append(tuple(traced_value.shape))
constructed_trace_agrees = constructed_shapes == crop_trace

parameter_before = {
    name: parameter.detach().clone() for name, parameter in model.named_parameters()
}
optimizer = torch.optim.Adam(model.parameters(), lr=0.025)
criterion = nn.CrossEntropyLoss()
loss_history = []
for _ in range(16):
    optimizer.zero_grad(set_to_none=True)
    logits = model(train_X)
    loss = criterion(logits, train_y)
    loss_history.append(float(loss.detach()))
    loss.backward()
    optimizer.step()

model_parameter_ids = {id(parameter) for parameter in model.parameters()}
optimizer_parameter_objects = [
    parameter for group in optimizer.param_groups for parameter in group["params"]
]
optimizer_parameter_ids = {id(parameter) for parameter in optimizer_parameter_objects}
optimizer_owns_exactly_model = (
    optimizer_parameter_ids == model_parameter_ids
    and len(optimizer_parameter_objects) == len(list(model.parameters()))
)
gradient_names = sorted(
    name for name, parameter in model.named_parameters() if parameter.grad is not None
)
moved_parameter_names = sorted(
    name
    for name, parameter in model.named_parameters()
    if not torch.equal(parameter.detach(), parameter_before[name])
)
training_certificate = bool(
    constructed_trace_agrees
    and tuple(logits.shape) == (12, 4)
    and optimizer_owns_exactly_model
    and gradient_names == sorted(name for name, _ in model.named_parameters())
    and moved_parameter_names
    and len(loss_history) == 16
    and torch.isfinite(torch.tensor(loss_history)).all()
    and loss_history[-1] <= 0.75 * loss_history[0]
)

### Answer check

In [ ]:
def floor_output(size, kernel, stride, padding):
    return (size + 2 * padding - kernel) // stride + 1

def independent_trace(batch, channels, height, width):
    height = floor_output(height, 5, 2, 2)
    width = floor_output(width, 5, 2, 2)
    first = (batch, 13, height, width)
    height = floor_output(height, 3, 2, 1)
    width = floor_output(width, 3, 2, 1)
    second = (batch, 13, height, width)
    height = floor_output(height, 3, 1, 1)
    width = floor_output(width, 5, 2, 0)
    return [first, second, (batch, 23, height, width)]

assert shape_trace == independent_trace(2, 7, 211, 229)
assert shape_trace == [
    (2, 13, 106, 115),
    (2, 13, 53, 58),
    (2, 23, 53, 27),
]
assert trace_ok is True
assert crop_trace == independent_trace(12, 7, 31, 35)
assert constructed_shapes == crop_trace and constructed_trace_agrees
assert isinstance(features[0], nn.Conv2d) and isinstance(features[1], nn.MaxPool2d)
assert isinstance(features[2], nn.Conv2d)
assert (features[0].in_channels, features[0].out_channels) == (7, 13)
assert features[0].kernel_size == (5, 5) and features[0].stride == (2, 2) and features[0].padding == (2, 2)
assert features[1].kernel_size == 3 and features[1].stride == 2 and features[1].padding == 1
assert (features[2].in_channels, features[2].out_channels) == (13, 23)
assert features[2].kernel_size == (3, 5) and features[2].stride == (1, 2) and features[2].padding == (1, 0)
assert isinstance(model[1], nn.AdaptiveAvgPool2d) and model[1].output_size == (2, 2)
assert isinstance(model[2], nn.Flatten) and model[2].start_dim == 1
assert isinstance(model[3], nn.Linear) and (model[3].in_features, model[3].out_features) == (92, 4)
assert tuple(logits.shape) == (12, 4)
assert optimizer_parameter_ids == {id(parameter) for parameter in model.parameters()}
assert len(optimizer_parameter_objects) == len(list(model.parameters()))
assert optimizer_owns_exactly_model
assert gradient_names == sorted(name for name, _ in model.named_parameters())
assert moved_parameter_names
assert set(moved_parameter_names).issubset(dict(model.named_parameters()))
assert len(loss_history) == 16
assert torch.isfinite(torch.tensor(loss_history)).all()
assert loss_history[-1] <= 0.75 * loss_history[0]
assert training_certificate